# Data Storm 2026: Latent Outlet Potential Pipeline

This notebook implements a notebook-first solution for the challenge:

1. Bronze ingestion of raw files.
2. Reusable data quality checks.
3. Silver cleaned datasets and rejected record stores.
4. Gold outlet-level feature engineering.
5. Latent potential estimation for January 2026.
6. Final prediction CSV and data quality report.

## 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
DATA_DIR = PROJECT_ROOT / 'data'
BRONZE_DIR = DATA_DIR / 'bronze'
SILVER_DIR = DATA_DIR / 'silver'
SILVER_REJECTED_DIR = DATA_DIR / 'silver_rejected'
GOLD_DIR = DATA_DIR / 'gold'
RESULTS_DIR = PROJECT_ROOT / 'Results'
DOCS_DIR = PROJECT_ROOT / 'Docs'

for directory in [BRONZE_DIR, SILVER_DIR, SILVER_REJECTED_DIR, GOLD_DIR, RESULTS_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    'outlet_master': 'outlet_master.csv',
    'outlet_coordinates': 'outlet_coordinates.csv',
    'transactions_history': 'transactions_history_final.csv',
    'distributor_seasonality': 'distributor_seasonality_details.csv',
    'holiday_list': 'holiday_list.csv',
}

VALID_DISTRIBUTOR_IDS = {
    'DIST_W_01', 'DIST_W_02', 'DIST_W_03',
    'DIST_C_01', 'DIST_C_02', 'DIST_C_03',
    'DIST_NW_01', 'DIST_NW_02',
    'DIST_S_01', 'DIST_S_02',
}

OUTLET_TYPE_MAP = {'Grocry': 'Grocery', 'Bakry': 'Bakery', 'Eatery ': 'Eatery'}
OUTLET_SIZE_MAP = {'small': 'Small', '': 'Unknown'}
VALID_OUTLET_TYPES = {'Grocery', 'Hotel', 'Pharmacy', 'SMMT', 'Kiosk', 'Bakery', 'Eatery'}
VALID_OUTLET_SIZES = {'Small', 'Medium', 'Large', 'Extra Large', 'Unknown'}
SEASONALITY_SCORE = {'Un-Favorable': -1, 'Moderate': 0, 'Favorable': 1}
RANDOM_STATE = 2026

print(f'Project root: {PROJECT_ROOT}')

## 2. Bronze Ingestion

Bronze keeps exact copies of the source files. The only additional artifact is an ingestion audit table.

In [ ]:
def file_checksum(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def ingest_bronze() -> pd.DataFrame:
    metadata = []
    ingested_at = datetime.now(timezone.utc).isoformat()
    for dataset_name, file_name in SOURCE_FILES.items():
        source = DATASETS_DIR / file_name
        target = BRONZE_DIR / file_name
        if not source.exists():
            raise FileNotFoundError(f'Missing required source file: {source}')
        shutil.copy2(source, target)
        metadata.append({
            'dataset_name': dataset_name,
            'source_file': str(source.relative_to(PROJECT_ROOT)),
            'bronze_file': str(target.relative_to(PROJECT_ROOT)),
            'row_count': len(pd.read_csv(target)),
            'sha256': file_checksum(target),
            'ingested_at_utc': ingested_at,
        })
    audit = pd.DataFrame(metadata)
    audit.to_csv(BRONZE_DIR / 'ingestion_audit.csv', index=False)
    return audit


bronze_audit = ingest_bronze()
bronze_audit

## 3. Reusable Data Quality Checks

In [ ]:
@dataclass(frozen=True)
class QualityResult:
    name: str
    valid_mask: pd.Series
    failure_reason: str


def duplicate_check(df: pd.DataFrame, key_columns: list[str]) -> QualityResult:
    return QualityResult(
        'duplicate_check',
        ~df.duplicated(subset=key_columns, keep='first'),
        f'Duplicate key on {", ".join(key_columns)}',
    )


def null_check(df: pd.DataFrame, columns: list[str]) -> QualityResult:
    valid = pd.Series(True, index=df.index)
    for column in columns:
        value = df[column]
        if pd.api.types.is_string_dtype(value) or value.dtype == object:
            valid &= value.astype('string').str.strip().ne('').fillna(False)
        else:
            valid &= value.notna()
    return QualityResult('null_check', valid, f'Null or empty mandatory field in {", ".join(columns)}')


def range_check(df: pd.DataFrame, column: str, min_value=None, max_value=None, inclusive=True) -> QualityResult:
    numeric = pd.to_numeric(df[column], errors='coerce')
    valid = numeric.notna()
    if min_value is not None:
        valid &= numeric.ge(min_value) if inclusive else numeric.gt(min_value)
    if max_value is not None:
        valid &= numeric.le(max_value) if inclusive else numeric.lt(max_value)
    return QualityResult('range_check', valid, f'{column} outside expected range')


def domain_check(df: pd.DataFrame, column: str, allowed_values: set[str]) -> QualityResult:
    valid = df[column].astype('string').str.strip().isin(allowed_values)
    return QualityResult('domain_check', valid, f'{column} contains a value outside the allowed domain')


def referential_integrity_check(df: pd.DataFrame, column: str, reference_values: Iterable[str]) -> QualityResult:
    valid = df[column].astype('string').isin(set(reference_values))
    return QualityResult('referential_integrity_check', valid, f'{column} does not exist in reference dataset')


def collect_rejections(df: pd.DataFrame, dataset_name: str, checks: list[QualityResult]):
    combined_valid = pd.Series(True, index=df.index)
    rejected_frames = []
    summary_rows = []
    for check in checks:
        valid = check.valid_mask.reindex(df.index).fillna(False).astype(bool)
        failed = ~valid
        combined_valid &= valid
        summary_rows.append({
            'dataset_name': dataset_name,
            'check_name': check.name,
            'failed_records': int(failed.sum()),
            'failure_reason': check.failure_reason,
        })
        if failed.any():
            rejected = df.loc[failed].copy()
            rejected.insert(0, 'failure_reason', check.failure_reason)
            rejected.insert(0, 'failed_check', check.name)
            rejected.insert(0, 'dataset_name', dataset_name)
            rejected_frames.append(rejected)
    valid_df = df.loc[combined_valid].copy()
    rejected_df = pd.concat(rejected_frames, ignore_index=True, sort=False) if rejected_frames else pd.DataFrame()
    return valid_df, rejected_df, pd.DataFrame(summary_rows)


def write_layer(dataset_name: str, valid_df: pd.DataFrame, rejected_df: pd.DataFrame) -> None:
    valid_df.to_csv(SILVER_DIR / f'{dataset_name}.csv', index=False)
    if rejected_df.empty:
        rejected_df = pd.DataFrame(columns=['dataset_name', 'failed_check', 'failure_reason'])
    rejected_df.to_csv(SILVER_REJECTED_DIR / f'{dataset_name}_rejected.csv', index=False)

## 4. Silver Cleaning

In [ ]:
quality_summaries = []

# Outlet master
outlets = pd.read_csv(BRONZE_DIR / SOURCE_FILES['outlet_master'])
outlets['Outlet_ID'] = outlets['Outlet_ID'].astype('string').str.strip()
outlets['Outlet_Size'] = outlets['Outlet_Size'].astype('string').fillna('').str.strip().replace(OUTLET_SIZE_MAP).fillna('Unknown')
outlets['Outlet_Type'] = outlets['Outlet_Type'].astype('string').fillna('').str.strip().replace(OUTLET_TYPE_MAP)
outlets['Cooler_Count'] = pd.to_numeric(outlets['Cooler_Count'], errors='coerce')
checks = [
    duplicate_check(outlets, ['Outlet_ID']),
    null_check(outlets, ['Outlet_ID']),
    range_check(outlets, 'Cooler_Count', 0, 20),
    domain_check(outlets, 'Outlet_Size', VALID_OUTLET_SIZES),
    domain_check(outlets, 'Outlet_Type', VALID_OUTLET_TYPES),
]
outlets_silver, outlets_rejected, summary = collect_rejections(outlets, 'outlet_master', checks)
outlets_silver['Cooler_Count'] = outlets_silver['Cooler_Count'].astype(int)
write_layer('outlet_master', outlets_silver, outlets_rejected)
quality_summaries.append(summary)
valid_outlet_ids = set(outlets_silver['Outlet_ID'])

# Outlet coordinates
coords = pd.read_csv(BRONZE_DIR / SOURCE_FILES['outlet_coordinates'])
coords['Outlet_ID'] = coords['Outlet_ID'].astype('string').str.strip()
coords['Latitude'] = pd.to_numeric(coords['Latitude'], errors='coerce')
coords['Longitude'] = pd.to_numeric(coords['Longitude'], errors='coerce')
checks = [
    duplicate_check(coords, ['Outlet_ID']),
    null_check(coords, ['Outlet_ID', 'Latitude', 'Longitude']),
    referential_integrity_check(coords, 'Outlet_ID', valid_outlet_ids),
    range_check(coords, 'Latitude', 5.5, 10.2),
    range_check(coords, 'Longitude', 79.0, 82.1),
]
coords_silver, coords_rejected, summary = collect_rejections(coords, 'outlet_coordinates', checks)
write_layer('outlet_coordinates', coords_silver, coords_rejected)
quality_summaries.append(summary)

# Transactions
transactions = pd.read_csv(BRONZE_DIR / SOURCE_FILES['transactions_history'])
transactions['Outlet_ID'] = transactions['Outlet_ID'].astype('string').str.strip()
transactions['Distributor_ID'] = transactions['Distributor_ID'].astype('string').str.strip()
transactions['SKU_ID'] = transactions['SKU_ID'].astype('string').str.strip()
for column in ['Year', 'Month', 'Volume_Liters', 'Total_Bill_Value']:
    transactions[column] = pd.to_numeric(transactions[column], errors='coerce')
checks = [
    null_check(transactions, ['Outlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID']),
    referential_integrity_check(transactions, 'Outlet_ID', valid_outlet_ids),
    domain_check(transactions, 'Distributor_ID', VALID_DISTRIBUTOR_IDS),
    range_check(transactions, 'Year', 2023, 2025),
    range_check(transactions, 'Month', 1, 12),
    range_check(transactions, 'Volume_Liters', 0, None, inclusive=False),
    range_check(transactions, 'Total_Bill_Value', 0, None, inclusive=False),
]
transactions_silver, transactions_rejected, summary = collect_rejections(transactions, 'transactions_history', checks)
transactions_silver['Year'] = transactions_silver['Year'].astype(int)
transactions_silver['Month'] = transactions_silver['Month'].astype(int)
transactions_silver['Year_Month'] = pd.to_datetime(transactions_silver[['Year', 'Month']].assign(Day=1), errors='coerce')
write_layer('transactions_history', transactions_silver, transactions_rejected)
quality_summaries.append(summary)

# Distributor seasonality
seasonality = pd.read_csv(BRONZE_DIR / SOURCE_FILES['distributor_seasonality'])
seasonality['Distributor_ID'] = seasonality['Distributor_ID'].astype('string').str.strip()
seasonality['Seasonality_Index'] = seasonality['Seasonality_Index'].astype('string').str.strip()
seasonality['Year'] = pd.to_numeric(seasonality['Year'], errors='coerce')
seasonality['Month'] = pd.to_numeric(seasonality['Month'], errors='coerce')
checks = [
    duplicate_check(seasonality, ['Distributor_ID', 'Year', 'Month']),
    domain_check(seasonality, 'Distributor_ID', VALID_DISTRIBUTOR_IDS),
    range_check(seasonality, 'Year', 2023, 2025),
    range_check(seasonality, 'Month', 1, 12),
    domain_check(seasonality, 'Seasonality_Index', set(SEASONALITY_SCORE)),
]
seasonality_silver, seasonality_rejected, summary = collect_rejections(seasonality, 'distributor_seasonality', checks)
seasonality_silver['Year'] = seasonality_silver['Year'].astype(int)
seasonality_silver['Month'] = seasonality_silver['Month'].astype(int)
seasonality_silver['Seasonality_Score'] = seasonality_silver['Seasonality_Index'].map(SEASONALITY_SCORE)
write_layer('distributor_seasonality', seasonality_silver, seasonality_rejected)
quality_summaries.append(summary)

# Holidays
holidays = pd.read_csv(BRONZE_DIR / SOURCE_FILES['holiday_list'])
holidays['Date'] = pd.to_datetime(holidays['Date'], errors='coerce', utc=True)
holidays['Holiday_Name'] = holidays['Holiday_Name'].astype('string').str.strip()
holidays['Holiday_Type'] = holidays['Holiday_Type'].astype('string').str.strip()
holidays['Year'] = holidays['Date'].dt.year
holidays['Month'] = holidays['Date'].dt.month
checks = [
    null_check(holidays, ['Date', 'Holiday_Name', 'Holiday_Type']),
    range_check(holidays, 'Year', 2023, 2026),
    range_check(holidays, 'Month', 1, 12),
]
holidays_silver, holidays_rejected, summary = collect_rejections(holidays, 'holiday_list', checks)
holidays_silver['Year'] = holidays_silver['Year'].astype(int)
holidays_silver['Month'] = holidays_silver['Month'].astype(int)
holidays_silver['Date'] = holidays_silver['Date'].dt.strftime('%Y-%m-%d')
write_layer('holiday_list', holidays_silver, holidays_rejected)
quality_summaries.append(summary)

quality_summary = pd.concat(quality_summaries, ignore_index=True)
quality_summary.to_csv(SILVER_REJECTED_DIR / 'quality_summary.csv', index=False)
quality_summary

## 5. Data Quality Report

In [ ]:
report_lines = [
    '# Data Quality Report',
    '',
    'This report is generated from `Notebooks/01_latent_potential_pipeline.ipynb`.',
    '',
    '## Check Summary',
    '',
    '| Dataset | Check | Failed Records | Failure Reason |',
    '| --- | --- | ---: | --- |',
]
for row in quality_summary.itertuples(index=False):
    report_lines.append(f'| `{row.dataset_name}` | `{row.check_name}` | {row.failed_records} | {row.failure_reason} |')

report_lines.extend(['', '## Rejected Record Stores', '', '| File | Rejected Rows |', '| --- | ---: |'])
for path in sorted(SILVER_REJECTED_DIR.glob('*_rejected.csv')):
    report_lines.append(f'| `{path.name}` | {len(pd.read_csv(path))} |')

(DOCS_DIR / 'data_quality_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
print(DOCS_DIR / 'data_quality_report.md')

## 6. Gold Feature Engineering

In [ ]:
SIZE_SCORE = {'Unknown': 0, 'Small': 1, 'Medium': 2, 'Large': 3, 'Extra Large': 4}
TYPE_SCORE = {'Kiosk': 0.80, 'Pharmacy': 0.85, 'Bakery': 1.00, 'Grocery': 1.10, 'Eatery': 1.15, 'Hotel': 1.20, 'SMMT': 1.25}

monthly = transactions_silver.groupby(['Outlet_ID', 'Year', 'Month'], as_index=False).agg(
    Monthly_Liters=('Volume_Liters', 'sum'),
    Monthly_Bill_Value=('Total_Bill_Value', 'sum'),
    SKU_Count=('SKU_ID', 'nunique'),
    Transaction_Lines=('SKU_ID', 'size'),
    Distributor_ID=('Distributor_ID', lambda values: values.mode().iat[0]),
)
monthly['Value_Per_Liter'] = (monthly['Monthly_Bill_Value'] / monthly['Monthly_Liters']).replace([np.inf, -np.inf], np.nan)
monthly.to_csv(GOLD_DIR / 'outlet_monthly_sales.csv', index=False)

sales_agg = monthly.groupby('Outlet_ID').agg(
    observed_mean_monthly_liters=('Monthly_Liters', 'mean'),
    observed_median_monthly_liters=('Monthly_Liters', 'median'),
    observed_max_monthly_liters=('Monthly_Liters', 'max'),
    observed_std_monthly_liters=('Monthly_Liters', 'std'),
    active_months=('Monthly_Liters', 'size'),
    mean_monthly_bill_value=('Monthly_Bill_Value', 'mean'),
    mean_sku_count=('SKU_Count', 'mean'),
    max_sku_count=('SKU_Count', 'max'),
    mean_transaction_lines=('Transaction_Lines', 'mean'),
    mean_value_per_liter=('Value_Per_Liter', 'mean'),
).reset_index()

jan = monthly.loc[monthly['Month'] == 1].groupby('Outlet_ID').agg(
    january_mean_liters=('Monthly_Liters', 'mean'),
    january_max_liters=('Monthly_Liters', 'max'),
).reset_index()

recent = monthly.loc[(monthly['Year'] == 2025) & (monthly['Month'].isin([10, 11, 12]))].groupby('Outlet_ID').agg(
    recent_3_month_mean_liters=('Monthly_Liters', 'mean'),
    recent_3_month_max_liters=('Monthly_Liters', 'max'),
).reset_index()

distributor_counts = monthly.groupby(['Outlet_ID', 'Distributor_ID']).size().rename('Distributor_Months').reset_index()
dominant_distributor = distributor_counts.sort_values(['Outlet_ID', 'Distributor_Months'], ascending=[True, False]).drop_duplicates('Outlet_ID')[['Outlet_ID', 'Distributor_ID']]

features = outlets_silver.merge(sales_agg, on='Outlet_ID', how='left')
features = features.merge(jan, on='Outlet_ID', how='left')
features = features.merge(recent, on='Outlet_ID', how='left')
features = features.merge(dominant_distributor, on='Outlet_ID', how='left')

coords_silver['has_valid_coordinates'] = 1
features = features.merge(coords_silver, on='Outlet_ID', how='left')
features['has_valid_coordinates'] = features['has_valid_coordinates'].fillna(0)
features['Latitude'] = features['Latitude'].fillna(features['Latitude'].median())
features['Longitude'] = features['Longitude'].fillna(features['Longitude'].median())

target_seasonality = seasonality_silver.loc[seasonality_silver['Month'] == 1].groupby('Distributor_ID', as_index=False)['Seasonality_Score'].mean().rename(columns={'Seasonality_Score': 'target_jan_seasonality_score'})
features = features.merge(target_seasonality, on='Distributor_ID', how='left')

holiday_months = holidays_silver.groupby(['Year', 'Month']).size().rename('holiday_count').reset_index()
features['target_jan_holiday_count'] = float(holiday_months.loc[holiday_months['Month'] == 1, 'holiday_count'].mean())
features['Outlet_Size_Score'] = features['Outlet_Size'].map(SIZE_SCORE).fillna(0)
features['Outlet_Type_Score'] = features['Outlet_Type'].map(TYPE_SCORE).fillna(1.0)
features['structural_capacity_score'] = (1 + 0.18 * features['Outlet_Size_Score'] + 0.10 * features['Cooler_Count']) * features['Outlet_Type_Score']

numeric_columns = features.select_dtypes(include=[np.number]).columns
features[numeric_columns] = features[numeric_columns].fillna(0)
features['Distributor_ID'] = features['Distributor_ID'].fillna('UNKNOWN')
features.to_csv(GOLD_DIR / 'outlet_features.csv', index=False)
features.head()

## 7. Latent Potential Model

The model blends a predicted observed baseline with a 90th-percentile demand frontier. The blend weight is a constraint score built from demand-proxy versus observed-performance gaps.

In [ ]:
FEATURE_COLUMNS = [
    'Outlet_Size', 'Outlet_Type', 'Cooler_Count', 'Latitude', 'Longitude',
    'has_valid_coordinates', 'mean_sku_count', 'max_sku_count',
    'mean_transaction_lines', 'mean_value_per_liter',
    'target_jan_seasonality_score', 'target_jan_holiday_count',
    'Outlet_Size_Score', 'Outlet_Type_Score', 'structural_capacity_score',
]

train_static = features[['Outlet_ID'] + FEATURE_COLUMNS]
train = monthly.merge(train_static, on='Outlet_ID', how='inner')
train_x = train[FEATURE_COLUMNS]
train_y = train['Monthly_Liters']

categorical = ['Outlet_Size', 'Outlet_Type']
numeric = [column for column in FEATURE_COLUMNS if column not in categorical]
def make_preprocessor():
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
        ('num', 'passthrough', numeric),
    ])

preprocessor = make_preprocessor()
frontier_preprocessor = make_preprocessor()

baseline_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingRegressor(max_iter=180, learning_rate=0.06, l2_regularization=0.05, random_state=RANDOM_STATE)),
])
baseline_model.fit(train_x, train_y)

frontier_model = Pipeline([
    ('preprocess', frontier_preprocessor),
    ('model', HistGradientBoostingRegressor(loss='quantile', quantile=0.90, max_iter=220, learning_rate=0.05, l2_regularization=0.05, random_state=RANDOM_STATE)),
])
frontier_model.fit(train_x, train_y)

features['observed_baseline_liters'] = baseline_model.predict(features[FEATURE_COLUMNS])
features['demand_frontier_liters'] = frontier_model.predict(features[FEATURE_COLUMNS])

demand_proxy = (
    0.45 * features['structural_capacity_score'].rank(pct=True)
    + 0.20 * features['Cooler_Count'].rank(pct=True)
    + 0.20 * features['mean_sku_count'].rank(pct=True)
    + 0.15 * features['has_valid_coordinates'].rank(pct=True)
)
observed_proxy = features['observed_mean_monthly_liters'].rank(pct=True)
gap = (demand_proxy - observed_proxy).clip(lower=0)
plateau = features['observed_std_monthly_liters'].fillna(0) / (features['observed_mean_monthly_liters'].fillna(0) + 1)
plateau_signal = (1 - plateau.rank(pct=True)).clip(0, 1)
features['constraint_score'] = (0.75 * gap + 0.25 * plateau_signal).clip(0, 1)

lower_bound = features[['observed_max_monthly_liters', 'january_max_liters', 'recent_3_month_max_liters']].max(axis=1)
peer_frontier = features.groupby(['Outlet_Type', 'Outlet_Size'])['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.90))
type_frontier = features.groupby('Outlet_Type')['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.85))
size_frontier = features.groupby('Outlet_Size')['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.85))
features['peer_frontier_liters'] = np.maximum.reduce([
    peer_frontier.fillna(0),
    type_frontier.fillna(0) * 0.85,
    size_frontier.fillna(0),
    features['demand_frontier_liters'],
    lower_bound,
])
frontier_gap = (features['peer_frontier_liters'] - lower_bound).clip(lower=0)
uncap_weight = (features['constraint_score'] ** 1.25).clip(0, 0.65)
raw_potential = lower_bound + uncap_weight * frontier_gap
peer_cap = features.groupby(['Outlet_Type', 'Outlet_Size'])['observed_max_monthly_liters'].transform(lambda s: s.quantile(0.98)).fillna(features['observed_max_monthly_liters'].quantile(0.98))
max_uplift_ratio = features['Outlet_Size'].map({'Unknown': 2.0, 'Small': 3.0, 'Medium': 3.5, 'Large': 4.0, 'Extra Large': 4.5}).fillna(3.0)
uplift_cap = lower_bound * max_uplift_ratio
soft_cap = np.maximum(np.minimum(peer_cap * 1.35, uplift_cap), lower_bound)
features['Maximum_Monthly_Liters'] = np.minimum(np.maximum(raw_potential, lower_bound), soft_cap).clip(lower=0).round(3)

predictions = features[['Outlet_ID', 'Maximum_Monthly_Liters']].copy()
predictions = predictions.rename(columns={'Outlet_ID': 'row_id'})
full_predictions = predictions.copy()
full_predictions.to_csv(RESULTS_DIR / 'teamname_predictions_full_20000.csv', index=False)

template_candidates = [
    DATASETS_DIR / 'sample_submission.csv',
    DATASETS_DIR / 'submission_template.csv',
    DATASETS_DIR / 'test.csv',
]
template_path = next((path for path in template_candidates if path.exists()), None)

if template_path is not None:
    template = pd.read_csv(template_path)
    if 'row_id' in template.columns and template['row_id'].isin(full_predictions['row_id']).any():
        predictions = template[['row_id']].merge(full_predictions, on='row_id', how='left')
    elif 'Outlet_ID' in template.columns:
        predictions = template[['Outlet_ID']].rename(columns={'Outlet_ID': 'row_id'}).merge(full_predictions, on='row_id', how='left')
    else:
        raise ValueError(f'Submission template {template_path} must contain row_id or Outlet_ID')
    if predictions['Maximum_Monthly_Liters'].isna().any():
        missing = predictions.loc[predictions['Maximum_Monthly_Liters'].isna(), 'row_id'].head().tolist()
        raise ValueError(f'Missing predictions for template row_id values: {missing}')
    submission_source = f'official template: {template_path.name}'
else:
    predictions = full_predictions.sort_values('row_id').head(914).copy()
    submission_source = 'fallback first 914 outlet IDs; replace with official template when available'

predictions.to_csv(RESULTS_DIR / 'teamname_predictions.csv', index=False)
print(f'Submission source: {submission_source}')

diagnostics = features[['Outlet_ID', 'Outlet_Size', 'Outlet_Type', 'Distributor_ID', 'observed_mean_monthly_liters', 'observed_max_monthly_liters', 'observed_baseline_liters', 'demand_frontier_liters', 'peer_frontier_liters', 'constraint_score', 'Maximum_Monthly_Liters']].copy()
diagnostics['uplift_ratio_vs_max'] = diagnostics['Maximum_Monthly_Liters'] / (diagnostics['observed_max_monthly_liters'] + 1)
diagnostics.to_csv(GOLD_DIR / 'prediction_diagnostics.csv', index=False)

predictions.head()

## 8. Final Validation

In [ ]:
validation = {
    'prediction_rows': len(predictions),
    'unique_outlets': predictions['row_id'].nunique(),
    'missing_predictions': int(predictions['Maximum_Monthly_Liters'].isna().sum()),
    'minimum_prediction': float(predictions['Maximum_Monthly_Liters'].min()),
    'median_prediction': float(predictions['Maximum_Monthly_Liters'].median()),
    'maximum_prediction': float(predictions['Maximum_Monthly_Liters'].max()),
    'output_file': str((RESULTS_DIR / 'teamname_predictions.csv').relative_to(PROJECT_ROOT)),
}
validation